# v5 비교 실험 — 같은 100명에게 voter_context_v5 + system_prompt_v5 적용

- 베이스: v3 (지역·세대 섹션 제거된 컨텍스트)
- 추가: 페르소나 카드에 `[정치적 이해관계]` 블록 주입 (`personas_gpt-5.4-mini_all_interests.tsv`의 `political_interest`)
- voter_context_v4: "투표 결정 요인"에 0번(정치적 이해관계) 1순위로 추가
- system_prompt_v4: 규칙 2번에 "정치적 이해관계 우선 인용" 한 줄 추가
- 그 외는 v3 동일
- 결과: `vote_results_all.csv`에 누적, raw는 `response/`

In [1]:
# [Cell 1] imports & env
import os, json, time, re
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

BASE = Path.cwd() if Path.cwd().name == "backend" else Path.cwd() / "backend"
CONTEXT_DIR = BASE / "context"
RESULTS_PATH = BASE / "vote_results_all.csv"
INTERESTS_PATH = BASE / "interests" / "personas_gpt-5.4-mini_all_interests.tsv"
RESPONSE_DIR = BASE / "response"
RESPONSE_DIR.mkdir(exist_ok=True)
LOG_DIR = BASE / "log"
LOG_DIR.mkdir(exist_ok=True)

MODEL = "gpt-5.4-mini"
VERSION = "v5"

for env_path in [BASE / ".env.local", BASE.parent / ".env.local"]:
    if env_path.exists():
        load_dotenv(env_path)
        print(f"loaded: {env_path}")
        break
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"

def safe_filename(s: str) -> str:
    return re.sub(r"[^\w\-.]", "_", s)

print(f"model: {MODEL}, version: {VERSION}")

loaded: z:\Github\persona-million\.env.local
model: gpt-5.4-mini, version: v5


In [2]:
# [Cell 2] v2 + gpt-5.4-mini 페르소나 100명 로드 + political_interest 매핑
df_all = pd.read_csv(RESULTS_PATH, encoding="utf-8-sig")
mask = (df_all["model"] == f"openai/{MODEL}") & (df_all["voter_context_version"] == "v2")
targets = df_all[mask].drop_duplicates(subset=["persona_uuid"]).reset_index(drop=True)
print(f"target personas: {len(targets)}")

interests = pd.read_csv(INTERESTS_PATH, sep="	", encoding="utf-8-sig")[["persona_uuid", "political_interest"]]
print(f"interests rows: {len(interests)}")

targets = targets.merge(interests, on="persona_uuid", how="left")
missing = targets["political_interest"].isna().sum()
print(f"political_interest 누락: {missing}/{len(targets)}")

# 이미 현재 VERSION 처리된 uuid 는 건너뛰기 (재실행 안전)
done = set(df_all[(df_all["model"] == f"openai/{MODEL}") & (df_all["voter_context_version"] == VERSION)]["persona_uuid"].tolist())
todo = targets[~targets["persona_uuid"].isin(done)].reset_index(drop=True)
print(f"이미 {VERSION} 처리됨: {len(done)}, 남은 작업: {len(todo)}")


target personas: 100
interests rows: 100
political_interest 누락: 0/100
이미 v5 처리됨: 0, 남은 작업: 100


In [3]:
# [Cell 3] v4 프롬프트 + 체인 + persona_card with political_interest
def load_context(version: str) -> str:
    return (CONTEXT_DIR / f"voter_context_{version}.md").read_text(encoding="utf-8")
def load_system_prompt(version: str) -> str:
    return (CONTEXT_DIR / f"system_prompt_{version}.md").read_text(encoding="utf-8")

POLITICAL_CONTEXT = load_context(VERSION)
SYSTEM_TEMPLATE = load_system_prompt(VERSION)
print(f"voter_context_{VERSION}.md: {len(POLITICAL_CONTEXT)} chars")
print(f"system_prompt_{VERSION}.md: {len(SYSTEM_TEMPLATE)} chars")

USER_TEMPLATE = """다음 페르소나가 2026년 6월 3일 지방선거(또는 가까운 미래의 총선·대선)에서 어느 정당을 지지·투표할지 추론하시오.

{persona_card}

JSON만 출력."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TEMPLATE),
    ("user", USER_TEMPLATE),
])
llm = ChatOpenAI(
    model=MODEL, temperature=0.7,
    model_kwargs={"response_format": {"type": "json_object"}},
)
chain = prompt | llm

VOTE_KEYS = ("vote", "party", "정당", "지지정당", "투표", "선택", "지지")
REASON_KEYS = ("reason", "이유", "rationale", "사유", "근거", "설명")

def _try_complete_json(s: str) -> str:
    last_close = s.rfind("}")
    last_open = s.rfind("{")
    if last_open == -1: return s
    if last_close > last_open: return s[last_open:last_close + 1]
    candidate = s[last_open:].rstrip().rstrip(",")
    if candidate.count('"') % 2 == 1: candidate += '"'
    candidate += "}"
    return candidate

def parse_response(raw: str) -> dict:
    s = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL)
    s = re.sub(r"```(?:json)?\s*", "", s).replace("```", "")
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if m:
        try: return json.loads(m.group(0))
        except json.JSONDecodeError: pass
    try: return json.loads(_try_complete_json(s))
    except json.JSONDecodeError:
        raise ValueError(f"no parseable JSON: {raw[:200]!r}")

def pick(d, keys):
    for k in keys:
        v = d.get(k)
        if v: return v
    return None

def persona_card(r) -> str:
    fields = [
        ("성별", r["sex"]), ("연령", f"{r['age']}세"),
        ("혼인상태", r["marital_status"]),
        ("가구형태", r["family_type"]), ("주거", r["housing_type"]),
        ("학력", r["education_level"]), ("전공", r["bachelors_field"]),
        ("직업", r["occupation"]),
        ("지역", f"{r['province']} {r['district']}"),
    ]
    demo = "\n".join(f"- {k}: {v}" for k, v in fields if v)
    narr = (
        f"[요약]\n{r['persona_summary']}\n\n"
        f"[직업적 면모]\n{r['professional_persona']}\n\n"
        f"[가족 면모]\n{r['family_persona']}\n\n"
        f"[문화적 배경]\n{r['cultural_background']}\n\n"
        f"[관심사]\n{r['hobbies_and_interests']}\n\n"
        f"[목표]\n{r['career_goals_and_ambitions']}"
    )
    out = f"## 인구통계\n{demo}\n\n## 인물 서사\n{narr}"
    pi = r.get("political_interest")
    if isinstance(pi, str) and pi.strip():
        out += f"\n\n## 정치적 이해관계\n[정치적 이해관계]\n{pi.strip()}"
    return out

# 샘플 프린트
if len(todo) > 0:
    print("\n--- 첫 페르소나 카드 미리보기 ---")
    print(persona_card(todo.iloc[0])[:1200])

voter_context_v5.md: 5475 chars
system_prompt_v5.md: 1778 chars

--- 첫 페르소나 카드 미리보기 ---
## 인구통계
- 성별: 여자
- 연령: 79세
- 혼인상태: 배우자있음
- 가구형태: 배우자와 거주
- 주거: 단독주택
- 학력: 초등학교
- 전공: 해당없음
- 직업: 건물 청소원
- 지역: 충청남 충청남-금산군

## 인물 서사
[요약]
장말순 씨는 금산 보건소의 보이지 않는 곳을 묵묵히 가꾸며, 소박한 노동과 정겨운 이웃 관계 속에서 삶의 안정을 찾는 79세의 강단 있는 할머니입니다.

[직업적 면모]
장말순 씨는 금산군 보건소 복도의 찌든 때를 없애기 위해 락스와 세제를 자신만의 비율로 섞어 사용하며, 내원객들이 붐비는 시간대를 피해 소리 없이 빠르게 바닥을 닦아내는 능숙한 청소원입니다.

[가족 면모]
장말순 씨는 금산의 낡은 단독주택에서 무뚝뚝하지만 평생 곁을 지켜준 남편과 함께 살며, 서로의 존재만으로도 충분한 고요하고 안정적인 노후를 보내고 있습니다.

[문화적 배경]
충남 금산의 인삼 밭이 내려다보이는 마을에서 나고 자라며 흙과 함께 유년 시절을 보냈습니다. 정규 교육은 짧았지만, 마을 공동체 안에서 서로 돕고 나누는 정서와 금산 특유의 느긋하면서도 강단 있는 생활 방식을 몸에 익혔습니다.

[관심사]
금강변 산책로를 천천히 걸으며 계절마다 변하는 풍경을 구경하거나, 동네 친구들과 함께 읍내 목욕탕에서 뜨끈한 물에 몸을 지지며 밀린 이야기를 나누는 시간을 보냅니다.

[목표]
몸이 허락하는 한 지금 하는 청소 일을 계속 유지하며, 매달 받는 월급으로 남편과 함께 가끔 외식을 즐기고 소소한 생활비를 보태는 안정적인 일상을 유지하고자 합니다.

## 정치적 이해관계
[정치적 이해관계]
올해 일흔아홉, 금산군 보건소에서 "락스와 세제를 자신만의 비율로 섞어" 복도를 닦고 있다. 새벽에 라디오를 틀면 2026년 기초연금이 단독가구 월 40만 원까지 오른다, 선정기준액이 247만 원으로 높아졌다는

In [4]:
# [Cell 4] 100명 v4 추론 → CSV 누적 + response/ 저장
log_path = LOG_DIR / f"{time.strftime('%Y%m%d-%H%M%S')}_voteV5_openai_{safe_filename(MODEL)}_{VERSION}.log"
log_f = log_path.open("w", encoding="utf-8")
def log(msg):
    print(msg, flush=True)
    log_f.write(msg + "\n"); log_f.flush()

log(f"=== openai/{MODEL} × {len(todo)} (version={VERSION}) ===")
success = 0
for i, r in todo.iterrows():
    t0 = time.perf_counter()
    raw = ""
    try:
        msg = chain.invoke({"political_context": POLITICAL_CONTEXT, "persona_card": persona_card(r)})
        raw = msg.content if hasattr(msg, "content") else str(msg)
        p = parse_response(raw)
        vote = pick(p, VOTE_KEYS)
        reason = pick(p, REASON_KEYS)
        if not vote or not reason:
            raise ValueError(f"missing keys: {list(p.keys())}")
    except Exception as e:
        el = time.perf_counter() - t0
        log(f"[{i+1:>3}/{len(todo)}] ERROR ({el:.1f}s): {str(e)[:120]}")
        ts = time.strftime("%Y%m%d-%H%M%S")
        fname = f"FAIL_{ts}_openai_{safe_filename(MODEL)}_{r['persona_uuid'][:8]}_{VERSION}.txt"
        (RESPONSE_DIR / fname).write_text(
            f"=== ERROR: {e}\n=== MODEL: openai/{MODEL}\n=== UUID: {r['persona_uuid']}\n\n{raw}",
            encoding="utf-8")
        continue
    el = time.perf_counter() - t0
    success += 1

    ts = time.strftime("%Y%m%d-%H%M%S")
    fname = f"{ts}_openai_{safe_filename(MODEL)}_{r['persona_uuid'][:8]}_{VERSION}_{VERSION}.json"
    out = {
        "timestamp": ts,
        "model": f"openai/{MODEL}",
        "voter_context_version": VERSION,
        "system_prompt_version": VERSION,
        "persona_uuid": r["persona_uuid"],
        "elapsed_sec": round(el, 3),
        "persona": {
            "sex": r["sex"], "age": int(r["age"]),
            "marital_status": r["marital_status"],
            "family_type": r["family_type"],
            "housing_type": r["housing_type"],
            "education_level": r["education_level"],
            "bachelors_field": r["bachelors_field"],
            "occupation": r["occupation"],
            "province": r["province"], "district": r["district"],
            "persona_summary": r["persona_summary"],
        },
        "political_interest": r.get("political_interest"),
        "raw_response": raw,
        "parsed": {"vote": vote, "reason": reason},
    }
    (RESPONSE_DIR / fname).write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

    rec = {
        "persona_uuid": r["persona_uuid"],
        "sex": r["sex"], "age": int(r["age"]),
        "marital_status": r["marital_status"],
        "family_type": r["family_type"], "housing_type": r["housing_type"],
        "education_level": r["education_level"],
        "bachelors_field": r["bachelors_field"],
        "occupation": r["occupation"],
        "province": r["province"], "district": r["district"],
        "persona_summary": r["persona_summary"],
        "vote": vote, "reason": reason,
        "model": f"openai/{MODEL}",
        "elapsed_sec": round(el, 3),
        "response_file": fname,
        "voter_context_version": VERSION,
        "system_prompt_version": VERSION,
        "professional_persona": r["professional_persona"],
        "sports_persona": r["sports_persona"],
        "arts_persona": r["arts_persona"],
        "travel_persona": r["travel_persona"],
        "culinary_persona": r["culinary_persona"],
        "family_persona": r["family_persona"],
        "cultural_background": r["cultural_background"],
        "skills_and_expertise": r["skills_and_expertise"],
        "hobbies_and_interests": r["hobbies_and_interests"],
        "career_goals_and_ambitions": r["career_goals_and_ambitions"],
    }
    df_rec = pd.DataFrame([rec])
    if RESULTS_PATH.exists():
        df_rec.to_csv(RESULTS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_rec.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")
    log(f"[{i+1:>3}/{len(todo)}] {r['sex']} {r['age']}세 {r['province']:<6} "
        f"{str(r['occupation'])[:14]:<14} → {vote[:10]:<10} ({el:.1f}s)")

log(f"\n결과: {success}/{len(todo)} 성공")
log_f.close()

=== openai/gpt-5.4-mini × 100 (version=v5) ===
[  1/100] 여자 79세 충청남    건물 청소원         → 더불어민주당     (4.3s)
[  2/100] 남자 29세 대구     무직             → 더불어민주당     (3.3s)
[  3/100] 남자 27세 서울     전직 금속제품 생산 관리자 → 더불어민주당     (2.7s)
[  4/100] 여자 71세 대전     무직             → 더불어민주당     (2.6s)
[  5/100] 여자 52세 부산     전동차 정비원        → 진보당        (2.8s)
[  6/100] 여자 20세 서울     무직             → 더불어민주당     (4.2s)
[  7/100] 여자 26세 제주     초등학교 교사        → 더불어민주당     (2.8s)
[  8/100] 남자 45세 경상남    용접기 조작원        → 더불어민주당     (2.9s)
[  9/100] 여자 63세 충청북    요양 보호사         → 더불어민주당     (2.8s)
[ 10/100] 여자 67세 경기     무직             → 더불어민주당     (4.2s)
[ 11/100] 여자 40세 강원     혼례 종사원         → 더불어민주당     (2.7s)
[ 12/100] 남자 48세 서울     마케팅 전문가        → 더불어민주당     (2.6s)
[ 13/100] 남자 45세 인천     그 외 택배원        → 더불어민주당     (2.5s)
[ 14/100] 남자 38세 충청남    한식 조리사         → 더불어민주당     (3.1s)
[ 15/100] 여자 26세 서울     보육교사           → 더불어민주당     (2.7s)
[ 16/100] 남자 53세 부산     대형 화물차 운전원     → 국민의힘       (2.5s)
[ 17/100]

In [5]:
# [Cell 5] v2 / v3 / v4 / v5 비교
df_all = pd.read_csv(RESULTS_PATH, encoding="utf-8-sig")
def pick_v(v):
    sub = df_all[(df_all["model"] == f"openai/{MODEL}") & (df_all["voter_context_version"] == v)][["persona_uuid", "vote"]]
    return sub.drop_duplicates("persona_uuid", keep="last").rename(columns={"vote": f"vote_{v}"})

m = pick_v("v2")
for v in ["v3", "v4", "v5"]:
    m = m.merge(pick_v(v), on="persona_uuid", how="outer")
print(f"비교 가능: {len(m)}명")

for v in ["v2", "v3", "v4", "v5"]:
    col = f"vote_{v}"
    if col in m.columns:
        print(f"=== {v} 분포 ===")
        print(m[col].value_counts())
        print()

if "vote_v4" in m.columns and "vote_v5" in m.columns:
    both = m.dropna(subset=["vote_v4", "vote_v5"])
    diff = (both["vote_v4"] != both["vote_v5"]).sum()
    print(f"v4 → v5 변경: {diff}/{len(both)} ({diff/len(both)*100:.1f}%)")
    print("=== v4 → v5 전이행렬 ===")
    print(pd.crosstab(both["vote_v4"], both["vote_v5"], margins=True))


비교 가능: 100명
=== v2 분포 ===
vote_v2
국민의힘      42
더불어민주당    40
무당층/기권    10
개혁신당       8
Name: count, dtype: int64

=== v3 분포 ===
vote_v3
더불어민주당    49
국민의힘      37
개혁신당       9
무당층/기권     5
Name: count, dtype: int64

=== v4 분포 ===
vote_v4
더불어민주당    79
국민의힘      11
개혁신당       8
무당층/기권     1
진보당        1
Name: count, dtype: int64

=== v5 분포 ===
vote_v5
더불어민주당    93
국민의힘       6
진보당        1
Name: count, dtype: int64

v4 → v5 변경: 15/100 (15.0%)
=== v4 → v5 전이행렬 ===
vote_v5  국민의힘  더불어민주당  진보당  All
vote_v4                        
개혁신당        1       7    0    8
국민의힘        5       6    0   11
더불어민주당      0      79    0   79
무당층/기권      0       1    0    1
진보당         0       0    1    1
All         6      93    1  100
